# Reflex Colab smoke test (free tier, ~5 min)

Proves this repo's collector + interchange on real NVIDIA hardware with **zero pip installs**. Open this notebook in Colab (Runtime > Change runtime type > GPU), then Run all.

What it does: probes the GPU, clones `MugiZer/reflex`, runs the stdlib-only test subset, and proves real hardware identity capture (the field every collected run is keyed on). Collection itself is the follow-up notebook.

What it does NOT do: install nsys, profile anything, spend quota. That's the next notebook.

In [1]:
# Cell 1 — environment probe. Fails fast on wrong/missing GPU.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import platform; print(platform.python_version())"
import json, pathlib
pathlib.Path('/content/reflex_runs').mkdir(exist_ok=True)
print('probe ok — continue only if a T4/P100 (or better) is listed above')

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
3.13.15
probe ok — continue only if a T4/P100 (or better) is listed above


In [2]:
# Cell 2 — clone (public repo, no auth needed).
!test -d reflex || git clone --depth 1 https://github.com/MugiZer/reflex.git
%cd reflex
!git log --oneline -1

Cloning into 'reflex'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 205 (delta 10), reused 158 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (205/205), 6.14 MiB | 13.47 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/reflex
d1142a9 (grafted, HEAD -> main, origin/main, origin/HEAD) Merge pull request #10 from MugiZer/reflex-pipeline


In [3]:
# Cell 3 — smoke tests. Stdlib only: no pip install, no GPU code paths.
# (Full suite needs sklearn etc. — install requirements only if you want it.)
!python -m pytest tests/test_reflex_ledger.py tests/test_fakegpu.py tests/test_collect.py -q

................................F...                                     [100%]
=================================== FAILURES ===================================
__________________ test_converted_bundle_boundary_documented ___________________

    def test_converted_bundle_boundary_documented():
        # Converted bundles flow through structural consumers without crashing,
        # and carry machine-readable coverage so counter/tensor/stall attribution
        # can check instead of assuming fakegpu vocabulary.
>       from reflex import reconstruct as _R, select as _S

tests/test_collect.py:430: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
reflex/select.py:19: in <module>
    from . import confidence as _conf
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

    """Ticket 07: confidence layer over tournament outputs.
    
    Pure post-hoc calibration + decision policy. Consumes fused rankings and
    voice scores o

In [4]:
# Cell 4 — REAL hardware identity. The money cell: proves the machine
# yields comparable identity before any collection is attempted.
from reflex.collect import nvidia_smi_identity
ident = nvidia_smi_identity()
print('identity:', ident)
assert ident['hardware'] != 'unknown', 'no GPU identity — aborting (see runbook)'
print('identity ok — this dict travels in every run manifest')

identity: {'device': 'Tesla T4', 'hardware': 'Tesla T4', 'driver': '580.82.07', 'cuda': '13.0', 'collector_version': 'collect-v1'}
identity ok — this dict travels in every run manifest


In [5]:
# Cell 5 — device contract (what Colab must provide per run):
#   device(fault: str, seed: int) -> {artifact_name: bytes}
# e.g. {'trace.json': <kineto JSON bytes>} and/or {'subset.db': <nsys sqlite bytes>}.
# The collector handles manifest/DONE/checksums/resume around it.
# Paste failures + the manifest JSON back to the repo owner.

In [6]:
# Cell 6 — Drive backup (optional). Stage locally first (done above),
# single copy out; never profile onto Drive directly.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil, datetime
    dst = '/content/drive/MyDrive/reflex-colab/%s' % datetime.date.today().isoformat()
    shutil.copytree('/content/reflex_runs', dst, dirs_exist_ok=True)
    print('backed up to', dst)
except Exception as exc:
    print('drive backup skipped:', type(exc).__name__, exc)

Mounted at /content/drive
backed up to /content/drive/MyDrive/reflex-colab/2026-09-06


## Next steps

1. Paste back: the identity dict from Cell 4, the pytest tail from Cell 3, and any failure.
2. Full collection matrix (all faults × seeds, <2h shards, checkpoint every run) is the follow-up notebook — it reuses `collect()` + `scan_todo()` resume, so a killed session loses at most one run.
3. Open in Colab directly: `https://colab.research.google.com/github/MugiZer/reflex/blob/main/colab/Reflex_Colab_Smoke.ipynb`